In [54]:
# Install
%pip install -r requirements.txt --upgrade
%pip install wbdata

  Using cached gspread-6.2.1-py3-none-any.whl.metadata (11 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [55]:
# Import 
import fooddatacentral as fdc
import pandas as pd
import numpy as np
from datascience import *
from scipy.optimize import linprog
import wbdata
import warnings 

apikey = "hhXDThEWfQoAua41aVdG81iaAHEhLCC0An6FqziG"
warnings.filterwarnings('ignore')

In [56]:
# [A] Dietary Reference Intake

# Reading Table
df_min = Table.read_table('Dietary Requirements - diet_minimums.csv')
df_max = Table.read_table('Dietary Requirements - diet_maximums.csv')

#Sex: Type F or M
def dri(age, sex):
    if age <= 3:
        sex = 'C'
        age_t = '1-3'
    elif 4 <= age <= 8:
        age_t = '4-8'
    elif 9 <= age <= 13:
        age_t = '9-13'
    elif 14 <= age <= 18:
        age_t = '14-18'
    elif 19 <= age <= 30:
        age_t = '19-30'
    elif 31 <= age <= 50:
        age_t = '31-50'
    else:
        age_t = '51+'

    column = sex + " " + age_t

    # Some nutrients does not have a maximum so it should be infinite
    # Using 999999 because later linprog function does not allow float('Inf')
    # But 999999 would be fair as it is a really big number
    data = {
            'Nutrition': np.insert(df_min.column("Nutrition"), 0, 'Sodium, Na') ,
            'max': np.append(df_max.column(column), [999999]*(len(df_min.column(column))-1)),
            'min': np.insert(df_min.column(column), 0, 0) 
            }

    return pd.DataFrame(data)


# Testings: 
dri(20, 'F')

,Nutrition,max,min
0,"Sodium, Na",2300,0.0
1,Energy,3100,2000.0
2,Protein,999999,46.0
3,"Fiber, total dietary",999999,28.0
4,"Folate, DFE",999999,400.0
5,"Calcium, Ca",999999,1000.0
6,"Carbohydrate, by difference",999999,130.0
7,"Iron, Fe",999999,18.0
8,"Magnesium, Mg",999999,310.0
9,Niacin,999999,14.0


In [57]:
# [A] Nutritional content of different foods

# Put an FDC ID HERE!
id =  [['chicken',2727569],['rice', 1999628],['pineapple',2346398]] 

def nutrition_table(id):
    # List of nutrient
    nutrient_df = pd.DataFrame({
                    'Nutrition': np.insert(df_min.column("Nutrition"), 0, 'Sodium, Na') ,
                    })
    
    for i in id:
        # Modifying data
        food_nutrient = fdc.nutrients(apikey,fdc_id=i[1])
        food_nutrient = food_nutrient.rename(index={"Energy (Atwater General Factors)": "Energy"})
        food_nutrient = food_nutrient.reset_index().rename(columns={'index': 'Nutrition','Quantity':i[0]}).drop("Units", axis=1)
        food_nutrient[food_nutrient.select_dtypes("number").columns] = food_nutrient.select_dtypes("number").clip(lower=0)
        
        
        nutrient_df = pd.merge(nutrient_df, food_nutrient, on='Nutrition', how='left')
        
    return nutrient_df.fillna(0)


# Testings: 
nutrition_table(id)

,Nutrition,chicken,rice,pineapple
0,"Sodium, Na",48.07000,0.936300,0.000000
1,Energy,126.90500,44.095000,60.111300
2,Protein,21.40625,2.414375,0.460938
3,"Fiber, total dietary",0.00000,4.166000,0.934600
4,"Folate, DFE",0.00000,0.000000,0.000000
5,"Calcium, Ca",6.89800,0.766300,12.500000
6,"Carbohydrate, by difference",0.00000,8.170625,14.091463
7,"Iron, Fe",0.35350,0.144300,0.054000
8,"Magnesium, Mg",25.40000,14.060000,13.380000
9,Niacin,0.00000,2.741000,0.227500


In [58]:
# [A] Solution - Part A: Producing 'A' 'B'

# Producing A_ub using the nutrition table generate from the function above\n",
def A_ub_function(df_nutrient_table):
    A_ub = []
    for i in range(len(df_nutrient_table)):
        A_ub.append(list(df_nutrient_table.iloc[i][1:]))

    for i in range(len(A_ub)):
        negative_a_ub = []
        for j in range(len(A_ub[0])):
            negative_a_ub.append(-A_ub[i][j])
        A_ub.append(negative_a_ub)

    return A_ub


# Producing A_ub using the google sheet we manually input\n",
df_nutrition = Table.read_table('Food_Local.csv')

def A_ub_sheet(df_sheet):
    A_ub = []
    for i in range(len(df_sheet[0])):
        A_ub.append(list(df_sheet.row(i))[2:]) # [2:] as exlduing the first column of food and the second column of units

    for i in range(len(A_ub)):
        negative_a_ub = []
        for j in range(len(A_ub[0])):
            negative_a_ub.append(-A_ub[i][j])
        A_ub.append(negative_a_ub)
    return A_ub

# Producing b_ub with given age and gender
def b_ub_age_gender(age, gender):
    restrictions = dri(20, 'F')
    b_ub = list(restrictions["max"]) + list(-restrictions["min"])
    return b_ub


# Producing the list of cost, assuming the cost is on the first row
df_price = Table.read_table('Price_Local.csv')

def cost_sheet(df):
    return list(df.row(0))

# Testings:
# A_ub_function(nutrition_table(id))
# A_ub_sheet(df_nutrition)
# b_ub_age_gender(20, 'F')

In [59]:
# [A] Solution - Part B: Solving the Problem with 'linprog'
# Exploring Local food (local only)

# defining 'A'
A_ub = A_ub_sheet(df_nutrition)

# defining cost
c = cost_sheet(df_price)

# defining 'b'
b_ub = b_ub_age_gender(20, 'F')

# Defining food_list
food_list = list(df_nutrition.labels)[2:] #[2:] becasue to exclude the nutrient column and unit column

# Assumption: A_ub, the cost and the list of food have the same order
def minimum_cost(A_ub, b_ub, c, food_list):
    # Bound for the x
    bounds = []
    for i in range(len(A_ub[0])):
        bounds.append((0, None))
    
    # Solving the problem and showing it in dataframe
    solution = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds)

    df = pd.DataFrame({'Food': food_list,
            'Quantity (100g)': solution.x})
    return df


minimum_cost(A_ub, b_ub, c, food_list) 

,Food,Quantity (100g)
0,"Cabbage (green, raw)",0.958180
1,Mango,0.000000
2,Chicken,0.000000
3,"Rice (white, long grain, unenriched, raw)",0.000000
4,Soybean Oil,1.122487
5,"Bread (white, enriched, unbleached)",0.000000
6,"Onion (yellow, raw)",0.000000
7,Raw Sugar,0.000000
8,Beans (Black),1.779426
9,Fish/tuna,0.000000


In [65]:
# Exploring Local and import 

df_nutrition_LI = Table.read_table('Food_Import_Local.csv')
df_price_LI = Table.read_table('Price_Import_Local.csv')

# defining 'A'
A_ub_LI = A_ub_sheet(df_nutrition_LI)

# defining cost
c_LI = cost_sheet(df_price_LI)

# defining 'b'
b_ub_LI = b_ub_age_gender(20, 'F')

# Defining food_list
food_list_LI = list(df_nutrition_LI.labels)[2:]

minimum_cost(A_ub_LI, b_ub_LI, c_LI, food_list_LI)

,Food,Quantity (100g)
0,"Cabbage (green, raw)",0.000000
1,Mango,0.000000
2,Chicken,0.000000
3,"Rice (white, long grain, unenriched, raw)",0.000000
4,Soybean Oil,1.080714
5,"Bread (white, enriched, unbleached)",0.000000
6,"Onion (yellow, raw)",0.000000
7,Raw Sugar,0.000000
8,Beans (Black),0.211583
9,Fish/tuna,0.000000


In [66]:
# [B] What is total cost for population of interest? - Part A: Population Function

# Population function from project 1

def population_1year(year,sex,age,place):    
    if sex == 'F':
        gender = "FE"
    else:
        gender = "MA"

    lower_range = age
    while (lower_range % 5) != 0:
        lower_range -= 1
    
    upper_range = age 
    while ((upper_range-4) % 5) != 0:
        upper_range += 1

    if lower_range < 10:
        lower_str = "0" + str(lower_range)
    else:
        lower_str = str(lower_range)

    if upper_range < 10:
        upper_str = "0" + str(upper_range)
    else:
        upper_str = str(upper_range)
        
    data = "SP.POP." + lower_str + upper_str + "." + gender
    
    variable_labels = {data:"Population"}
    world = wbdata.get_dataframe(variable_labels, country=place, parse_dates=True)
    
    filtered = world[world.index.year == year]
    if filtered.empty:
        return 0
    return filtered.iloc[0]['Population'] / 5


#Getting the population by year, gender, age range and place.
#Coded based on population_1year function
def population(year,sex,age_range,place):
    lower_range = age_range[0]
    higher_range = age_range[1]
    population = 0
    
    while higher_range >= lower_range:
        population += population_1year(year,sex,lower_range,place)
        lower_range += 1

    return population
        
'''
Instruction for using the population function:
1. Year is limited to be between 1960 to 2024.
2. Type either 'F' or 'M' for gender.
3. See "wbdata.get_countries()"  for country code
4. About age range: 
    a. Type (lower range, upper range)
        i. It is inclusive, which mean the range (15,19) include the population for both age 15 and age 19.
    b. lower range has to be: 0 - 78
    c. upper range has to be: 1 - 79 
    d. Upper range has to be greater than lower range
'''

#Example:
print(population_1year(2021, 'F',19,'HTI'))
print(population(2021, 'F', (15,19), 'HTI'))
print(population(2021, 'F', (43,78), 'HTI'))

113653.0
568265.0
1213870.4


In [68]:
# [B] What is total cost for population of interest? - Part B: Calculating Total cost 

# Total cost of the population of one age and one gender in one country on input year
def total_cost(year, gender, age, country, cost, A_ub, food_list):
    b_ub = b_ub_age_gender(age, gender)
    df_food = minimum_cost(A_ub, b_ub, cost, food_list)
    return sum(list(df_food['Quantity (100g)'] * cost)) * population_1year(year, gender, age, country)

# Total cost of the population of the input country on input year
def total_cost_whole_country(year, country, cost, A_ub, food_list):

    # Getting total cost for one gender
    def total_cost_one_gender(gender):

        # Adding population for 80UP as my population does not include that
        if gender == 'F':
            g = 'FE'
        else: 
            g = 'MA'

        data = "SP.POP.80UP." + g
    
        variable_labels = {data:"Population"}
        world = wbdata.get_dataframe(variable_labels, country=country, parse_dates=True)  
        filtered = world[world.index.year == year]
        total = filtered.iloc[0]['Population']

        for i in range(79):
            total += total_cost(year, gender, i, country, cost, A_ub,food_list)
        return total
        
    df = pd.DataFrame({'Gender': ['Male','Female','Total'],
            'Cost': [total_cost_one_gender('M'), total_cost_one_gender('F'), total_cost_one_gender('M')+total_cost_one_gender('F')]})
    return df

In [69]:
# Total cost for local food only
total_cost_whole_country(2024, 'HTI', c, A_ub, food_list)

,Gender,Cost
0,Male,1.374006e+07
1,Female,1.399755e+07
2,Total,2.773761e+07


In [70]:
# Total cost for having both local and import food
total_cost_whole_country(2024, 'HTI', c_LI, A_ub_LI, food_list_LI)

,Gender,Cost
0,Male,1.469968e+07
1,Female,1.497403e+07
2,Total,2.967371e+07


In [71]:
# [C] Sensitivity of solution 

# Cost of one meal of the person with input age and input gender
def one_meal_cost(gender, age, cost, A_ub, food_list):
    b_ub = b_ub_age_gender(age, gender)
    df_food = minimum_cost(A_ub, b_ub, cost, food_list)
    return sum(list(df_food['Quantity (100g)'] * cost))

# Sensitivity 
# Assumption: A_ub, the cost and the list of food have the same order
def sensitivity(gender, age, cost, A_ub, food_list, food_with_price_change):

    index = food_list.index(food_with_price_change)
    
    percentage_change = [0, 5, 10, 50, 100, 200]
    b_ub = b_ub_age_gender(age, gender)
    df = {'Food': food_list + ['Total Price']}
    
    for i in percentage_change:
        new_cost = cost[:]
        new_cost[index] *= (1+(0.01*i))
        df_food = minimum_cost(A_ub, b_ub, new_cost, food_list)
        df[str(i)+"% change for " + food_with_price_change] = list(df_food['Quantity (100g)']) + [one_meal_cost(gender, age, new_cost, A_ub, food_list)]

    return pd.DataFrame(df)
        
sensitivity('F', 20, c , A_ub,food_list, 'Mango')

,Food,0% change for Mango,5% change for Mango,10% change for Mango,50% change for Mango,100% change for Mango,200% change for Mango
0,"Cabbage (green, raw)",0.958180,0.958180,0.958180,0.958180,0.958180,0.958180
1,Mango,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Chicken,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,"Rice (white, long grain, unenriched, raw)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Soybean Oil,1.122487,1.122487,1.122487,1.122487,1.122487,1.122487
5,"Bread (white, enriched, unbleached)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,"Onion (yellow, raw)",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,Raw Sugar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,Beans (Black),1.779426,1.779426,1.779426,1.779426,1.779426,1.779426
9,Fish/tuna,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
